In [11]:
import pathlib as pl
import pickle as pck
import os

import pandas as pd

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)

# source for GRCh38
# ftp://ftp.ncbi.nlm.nih.gov
# /genomes/all/GCA/000/001/405/GCA_000001405.15_GRCh38/seqs_for_alignment_pipelines.ucsc_ids
# /GCA_000001405.15_GRCh38_no_alt_analysis_set.fna.gz

# source for T2Tv2/CHM13
# https://s3-us-west-2.amazonaws.com
# /human-pangenomics/T2T/CHM13/assemblies/analysis_set
# /chm13v2.0.fa.gz

import collections as col
import dnaio
import io

cache_folder = CONFIG["nb_cache_folder"]

grch38_ref_file = cache_folder.joinpath("GCA_000001405.15_GRCh38_no_alt_analysis_set.fna.gz").resolve(strict=True)
t2t_ref_file = cache_folder.joinpath("chm13v2.0.fa.gz").resolve(strict=True)

ref_files = [grch38_ref_file, t2t_ref_file]
ref_labels = ["hg38", "t2tv2"]

buffers = col.defaultdict(io.StringIO)
for ref_file, ref_label in zip(ref_files, ref_labels):
    with dnaio.open(ref_file) as ref:
        for record in ref:
            split_name = record.name.split()[0]
            if "chrX" in split_name:
                sample_name = ref_1kg_map[ref_label]
                buffers[(ref_label, "chrX")].write(
                    f">chrX_{sample_name}_{split_name}\n{record.sequence.upper()}\n"
                )
            if "chrY" in split_name:
                if "random" in split_name:
                    split_name = split_name.split("_")[1]
                sample_name = ref_1kg_map[ref_label]
                buffers[(ref_label, "chrY")].write(
                    f">chrY_{sample_name}_{split_name}\n{record.sequence.upper()}\n"
                )

ref_1kg_map = {
    "hg38": "RFGRC38-R1",
    "t2tv2": "RFCHM13-J1"
}

for (ref_label, chrom), buffer in buffers.items():
    sample = ref_1kg_map[ref_label]
    out_file = cache_folder.joinpath(f"{sample}.{chrom}.fasta")
    with open(out_file, "w") as fasta:
        _ = fasta.write(buffer.getvalue())
    
    